In [23]:
# ============================================================
# CEK MISSING VALUE - DATA MENTAH (DENGAN REPLACE)
# ============================================================

import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import numpy as np

CREDENTIALS_PATH = r"C:\Users\fardh\Skripsi\Folder Baru\bim-flight-demo\credentials\credentials.json"
GOOGLE_SHEET_KEY = "1njLUn55TWwYjTQKEmmR97bcwgYaLYdF-0j5vK-vUIRw"

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]
creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_PATH, scope)
gc = gspread.authorize(creds)
sheet = gc.open_by_key(GOOGLE_SHEET_KEY)

# ============================================================
# 1. BACA DATA
# ============================================================

df_arrival = pd.DataFrame(sheet.worksheet("RAW_ARRIVAL").get_all_records())
df_departure = pd.DataFrame(sheet.worksheet("RAW_DEPARTURE").get_all_records())

print("="*70)
print("CEK MISSING VALUE - DATA MENTAH (DENGAN REPLACE)")
print("="*70)

# ============================================================
# 2. REPLACE STRING KOSONG MENJADI NaN
# ============================================================

print("\nMengganti string kosong dengan NaN...")
df_arrival = df_arrival.replace(r'^\s*$', np.nan, regex=True)
df_departure = df_departure.replace(r'^\s*$', np.nan, regex=True)
print(" Selesai!")

# ============================================================
# 3. CEK DATA ARRIVAL
# ============================================================

print("\n" + "-"*70)
print("RAW_ARRIVAL")
print("-"*70)

print(f"Total baris : {len(df_arrival)}")
print(f"Total kolom : {len(df_arrival.columns)}")

# Missing value per kolom
missing_arr = df_arrival.isnull().sum()
missing_arr_pct = (missing_arr / len(df_arrival) * 100).round(2)

# Buat DataFrame untuk tampilan
df_missing_arr = pd.DataFrame({
    'Kolom': missing_arr.index,
    'Missing_Value': missing_arr.values,
    'Persentase (%)': missing_arr_pct.values
})

# Filter kolom yang memiliki missing value
df_missing_arr_has = df_missing_arr[df_missing_arr['Missing_Value'] > 0].sort_values('Missing_Value', ascending=False)

print("\nJumlah Missing Value per kolom:")
print(df_missing_arr_has.to_string(index=False))

if len(df_missing_arr_has) == 0:
    print(" Tidak ada missing value!")

# ============================================================
# 4. CEK DATA DEPARTURE
# ============================================================

print("\n" + "-"*70)
print(" RAW_DEPARTURE")
print("-"*70)

print(f"Total baris : {len(df_departure)}")
print(f"Total kolom : {len(df_departure.columns)}")

# Missing value per kolom
missing_dep = df_departure.isnull().sum()
missing_dep_pct = (missing_dep / len(df_departure) * 100).round(2)

# Buat DataFrame untuk tampilan
df_missing_dep = pd.DataFrame({
    'Kolom': missing_dep.index,
    'Missing_Value': missing_dep.values,
    'Persentase (%)': missing_dep_pct.values
})

# Filter kolom yang memiliki missing value
df_missing_dep_has = df_missing_dep[df_missing_dep['Missing_Value'] > 0].sort_values('Missing_Value', ascending=False)

print("\nJumlah Missing Value per kolom:")
print(df_missing_dep_has.to_string(index=False))

if len(df_missing_dep_has) == 0:
    print("Tidak ada missing value!")

# ============================================================
# 5. KOLOM KRITIS UNTUK PENELITIAN
# ============================================================

print("\n" + "="*70)
print("FOKUS: KOLOM KRITIS UNTUK PENELITIAN")
print("="*70)

critical_cols_arr = ["Landing_Time", "Onblock_Time", "STA"]
critical_cols_dep = ["BlockOff_Time", "TakeOff_Time", "STD"]

print("\n Arrival - Kolom kritis:")
for col in critical_cols_arr:
    if col in df_arrival.columns:
        missing = df_arrival[col].isnull().sum()
        pct = (missing / len(df_arrival) * 100).round(2)
        status = "Tidak ada Mising Value" if missing == 0 else "Mising Value"
        print(f"   {status} {col}: {missing} missing ({pct}%)")

print("\n Departure - Kolom kritis:")
for col in critical_cols_dep:
    if col in df_departure.columns:
        missing = df_departure[col].isnull().sum()
        pct = (missing / len(df_departure) * 100).round(2)
        status = "Tidak ada Mising Value" if missing == 0 else "Mising Value"
        print(f"   {status} {col}: {missing} missing ({pct}%)")

# ============================================================
# 6. RINGKASAN AKHIR
# ============================================================

print("\n" + "="*70)
print("RINGKASAN DATA")
print("="*70)

print(f"""
RAW_ARRIVAL:
   - Total baris   : {len(df_arrival)}
   - Total missing : {df_arrival.isnull().sum().sum()} nilai
   - Kolom dengan missing : {len(df_missing_arr_has)} kolom

RAW_DEPARTURE:
   - Total baris   : {len(df_departure)}
   - Total missing : {df_departure.isnull().sum().sum()} nilai
   - Kolom dengan missing : {len(df_missing_dep_has)} kolom
""")

print("="*70)
print("CEK MISSING VALUE SELESAI")

CEK MISSING VALUE - DATA MENTAH (DENGAN REPLACE)

Mengganti string kosong dengan NaN...
 Selesai!

----------------------------------------------------------------------
RAW_ARRIVAL
----------------------------------------------------------------------
Total baris : 3498
Total kolom : 20

Jumlah Missing Value per kolom:
        Kolom  Missing_Value  Persentase (%)
Paired_Flight            453           12.95
 Onblock_Time             59            1.69
 Landing_Time             48            1.37
 Aircraft_Reg             38            1.09
        Stand             26            0.74
Aircraft_Type              1            0.03
       Origin              1            0.03

----------------------------------------------------------------------
 RAW_DEPARTURE
----------------------------------------------------------------------
Total baris : 3503
Total kolom : 20

Jumlah Missing Value per kolom:
        Kolom  Missing_Value  Persentase (%)
Paired_Flight            448           12.79
 

In [43]:
# ============================================================
# ANALISIS KARAKTERISTIK DATA MENTAH
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gspread
from oauth2client.service_account import ServiceAccountCredentials

CREDENTIALS_PATH = r"C:\Users\fardh\Skripsi\Folder Baru\bim-flight-demo\credentials\credentials.json"
GOOGLE_SHEET_KEY = "1njLUn55TWwYjTQKEmmR97bcwgYaLYdF-0j5vK-vUIRw"

scope = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive"
]
creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_PATH, scope)
gc = gspread.authorize(creds)
sheet = gc.open_by_key(GOOGLE_SHEET_KEY)

# Baca data mentah
df_arr = pd.DataFrame(sheet.worksheet("RAW_ARRIVAL").get_all_records())
df_dep = pd.DataFrame(sheet.worksheet("RAW_DEPARTURE").get_all_records())

# Ganti string kosong menjadi NaN
df_arr = df_arr.replace(r'^\s*$', np.nan, regex=True)
df_dep = df_dep.replace(r'^\s*$', np.nan, regex=True)

print("="*70)
print(" ANALISIS KARAKTERISTIK DATA MENTAH")
print("="*70)

# ============================================================
# 1. STRUKTUR DATA
# ============================================================

print("\n" + "="*70)
print("1. STRUKTUR DATA")
print("="*70)

print(f"\n RAW_ARRIVAL:")
print(f"   Jumlah baris : {len(df_arr)}")
print(f"   Jumlah kolom : {len(df_arr.columns)}")
print(f"   Nama kolom   : {df_arr.columns.tolist()}")

print(f"\n RAW_DEPARTURE:")
print(f"   Jumlah baris : {len(df_dep)}")
print(f"   Jumlah kolom : {len(df_dep.columns)}")
print(f"   Nama kolom   : {df_dep.columns.tolist()}")

# ============================================================
# 2. TIPE DATA
# ============================================================

print("\n" + "="*70)
print("2. TIPE DATA")
print("="*70)

print("\n RAW_ARRIVAL:")
print(df_arr.dtypes)

print("\n RAW_DEPARTURE:")
print(df_dep.dtypes)

# ============================================================
# 3. MISSING VALUE
# ============================================================

print("\n" + "="*70)
print("3. MISSING VALUE")
print("="*70)

# Arrival
missing_arr = df_arr.isnull().sum()
missing_arr_pct = (missing_arr / len(df_arr) * 100).round(2)
df_missing_arr = pd.DataFrame({
    'Kolom': missing_arr.index,
    'Missing_Value': missing_arr.values,
    'Persentase (%)': missing_arr_pct.values
})
df_missing_arr_has = df_missing_arr[df_missing_arr['Missing_Value'] > 0].sort_values('Missing_Value', ascending=False)

print("\nRAW_ARRIVAL - Missing Value:")
print(df_missing_arr_has.to_string(index=False))

# Departure
missing_dep = df_dep.isnull().sum()
missing_dep_pct = (missing_dep / len(df_dep) * 100).round(2)
df_missing_dep = pd.DataFrame({
    'Kolom': missing_dep.index,
    'Missing_Value': missing_dep.values,
    'Persentase (%)': missing_dep_pct.values
})
df_missing_dep_has = df_missing_dep[df_missing_dep['Missing_Value'] > 0].sort_values('Missing_Value', ascending=False)

print("\n RAW_DEPARTURE - Missing Value:")
print(df_missing_dep_has.to_string(index=False))

# ============================================================
# 4. EKSTRAKSI FITUR DASAR DARI DATA MENTAH
# ============================================================

print("\n" + "="*70)
print(" 4. EKSTRAKSI FITUR DASAR")
print("="*70)

# Ekstrak Airline dari Flight_Number
def extract_airline(flight):
    if pd.isna(flight):
        return np.nan
    return str(flight).strip().upper()[:2]

df_arr["Airline"] = df_arr["Flight_Number"].apply(extract_airline)
df_dep["Airline"] = df_dep["Flight_Number"].apply(extract_airline)

# Ekstrak Data_Date ke datetime
df_arr["Data_Date"] = pd.to_datetime(df_arr["Data_Date"], errors="coerce")
df_dep["Data_Date"] = pd.to_datetime(df_dep["Data_Date"], errors="coerce")

# Ekstrak Day_of_Week dan Month
df_arr["Day_of_Week"] = df_arr["Data_Date"].dt.day_name()
df_arr["Month"] = df_arr["Data_Date"].dt.month_name()
df_dep["Day_of_Week"] = df_dep["Data_Date"].dt.day_name()
df_dep["Month"] = df_dep["Data_Date"].dt.month_name()

print("Fitur dasar berhasil diekstrak")

# ============================================================
# 5. DISTRIBUSI MASKAPAI
# ============================================================

print("\n" + "="*70)
print(" 5. DISTRIBUSI MASKAPAI (AIRLINE)")
print("="*70)

airline_arr = df_arr["Airline"].value_counts()
airline_arr_pct = df_arr["Airline"].value_counts(normalize=True) * 100

airline_dep = df_dep["Airline"].value_counts()
airline_dep_pct = df_dep["Airline"].value_counts(normalize=True) * 100

print("\n RAW_ARRIVAL - Top 10 Maskapai:")
print("┌──────────────┬───────────┬─────────────┐")
print("│   Maskapai   │  Jumlah   │  Persentase │")
print("├──────────────┼───────────┼─────────────┤")
for idx, (airline, count) in enumerate(airline_arr.head(10).items()):
    pct = airline_arr_pct[airline]
    print(f"│ {airline:>12} │ {count:>9} │ {pct:>11.2f}% │")
print("└──────────────┴───────────┴─────────────┘")

print("\n RAW_DEPARTURE - Top 10 Maskapai:")
print("┌──────────────┬───────────┬─────────────┐")
print("│   Maskapai   │  Jumlah   │  Persentase │")
print("├──────────────┼───────────┼─────────────┤")
for idx, (airline, count) in enumerate(airline_dep.head(10).items()):
    pct = airline_dep_pct[airline]
    print(f"│ {airline:>12} │ {count:>9} │ {pct:>11.2f}% │")
print("└──────────────┴───────────┴─────────────┘")

# ============================================================
# 6. DISTRIBUSI JENIS PENERBANGAN
# ============================================================

print("\n" + "="*70)
print(" 6. DISTRIBUSI JENIS PENERBANGAN")
print("="*70)

print("\n RAW_ARRIVAL:")
print(df_arr["Table_Type"].value_counts().to_string())

print("\n RAW_DEPARTURE:")
print(df_dep["Table_Type"].value_counts().to_string())

# ============================================================
# 7. RENTANG TANGGAL OPERASIONAL
# ============================================================

print("\n" + "="*70)
print(" 7. RENTANG TANGGAL OPERASIONAL")
print("="*70)

print(f"\n RAW_ARRIVAL:")
print(f"   Tanggal awal  : {df_arr['Data_Date'].min()}")
print(f"   Tanggal akhir : {df_arr['Data_Date'].max()}")
print(f"   Total hari    : {(df_arr['Data_Date'].max() - df_arr['Data_Date'].min()).days + 1} hari")

print(f"\n RAW_DEPARTURE:")
print(f"   Tanggal awal  : {df_dep['Data_Date'].min()}")
print(f"   Tanggal akhir : {df_dep['Data_Date'].max()}")
print(f"   Total hari    : {(df_dep['Data_Date'].max() - df_dep['Data_Date'].min()).days + 1} hari")

# ============================================================
# 8. DISTRIBUSI HARI OPERASIONAL
# ============================================================

print("\n" + "="*70)
print(" 8. DISTRIBUSI HARI OPERASIONAL")
print("="*70)

day_arr = df_arr["Day_of_Week"].value_counts().sort_index()
day_arr_pct = df_arr["Day_of_Week"].value_counts(normalize=True).sort_index() * 100

day_dep = df_dep["Day_of_Week"].value_counts().sort_index()
day_dep_pct = df_dep["Day_of_Week"].value_counts(normalize=True).sort_index() * 100

print("\n RAW_ARRIVAL:")
print("┌─────────────┬───────────┬─────────────┐")
print("│     Hari    │  Jumlah   │  Persentase │")
print("├─────────────┼───────────┼─────────────┤")
for day in ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]:
    if day in day_arr.index:
        print(f"│ {day:>11} │ {day_arr[day]:>9} │ {day_arr_pct[day]:>11.2f}% │")
print("└─────────────┴───────────┴─────────────┘")

print("\n RAW_DEPARTURE:")
print("┌─────────────┬───────────┬─────────────┐")
print("│     Hari    │  Jumlah   │  Persentase │")
print("├─────────────┼───────────┼─────────────┤")
for day in ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]:
    if day in day_dep.index:
        print(f"│ {day:>11} │ {day_dep[day]:>9} │ {day_dep_pct[day]:>11.2f}% │")
print("└─────────────┴───────────┴─────────────┘")

# ============================================================
# 9. DISTRIBUSI BULAN OPERASIONAL
# ============================================================

print("\n" + "="*70)
print(" 9. DISTRIBUSI BULAN OPERASIONAL")
print("="*70)

month_arr = df_arr["Month"].value_counts().sort_index()
month_arr_pct = df_arr["Month"].value_counts(normalize=True).sort_index() * 100

month_dep = df_dep["Month"].value_counts().sort_index()
month_dep_pct = df_dep["Month"].value_counts(normalize=True).sort_index() * 100

print("\n RAW_ARRIVAL:")
for month in ["February", "March", "April", "May", "June"]:
    if month in month_arr.index:
        print(f"   {month}: {month_arr[month]} ({month_arr_pct[month]:.2f}%)")

print("\n RAW_DEPARTURE:")
for month in ["February", "March", "April", "May", "June"]:
    if month in month_dep.index:
        print(f"   {month}: {month_dep[month]} ({month_dep_pct[month]:.2f}%)")

# ============================================================
# 10. STATISTIK DESKRIPTIF - KOLOM NUMERIK (PAX, CARGO, BAG)
# ============================================================

print("\n" + "="*70)
print(" 10. STATISTIK DESKRIPTIF - KOLOM NUMERIK")
print("="*70)

numerik_cols = ["Adult_Pax", "Child_Pax", "Infant_Pax", "Total_Pax", "Seat", "Cargo_Kg", "Baggage_Kg"]

print("\n RAW_ARRIVAL:")
for col in numerik_cols:
    if col in df_arr.columns:
        print(f"\n   {col}:")
        print(f"      Mean  : {df_arr[col].mean():.2f}")
        print(f"      Median: {df_arr[col].median():.0f}")
        print(f"      Min   : {df_arr[col].min():.0f}")
        print(f"      Max   : {df_arr[col].max():.0f}")
        print(f"      Std   : {df_arr[col].std():.2f}")

print("\n RAW_DEPARTURE:")
for col in numerik_cols:
    if col in df_dep.columns:
        print(f"\n   {col}:")
        print(f"      Mean  : {df_dep[col].mean():.2f}")
        print(f"      Median: {df_dep[col].median():.0f}")
        print(f"      Min   : {df_dep[col].min():.0f}")
        print(f"      Max   : {df_dep[col].max():.0f}")
        print(f"      Std   : {df_dep[col].std():.2f}")

# ============================================================
# 11. SAMPLE DATA
# ============================================================

print("\n" + "="*70)
print(" 11. SAMPLE DATA")
print("="*70)

print("\n RAW_ARRIVAL - 5 baris pertama:")
print(df_arr[["Flight_Number", "Paired_Flight", "Aircraft_Reg", "Aircraft_Type", "Origin", "STA", "Landing_Time", "Onblock_Time", "Stand", "Data_Date"]].head())

print("\n RAW_DEPARTURE - 5 baris pertama:")
print(df_dep[["Flight_Number", "Paired_Flight", "Aircraft_Reg", "Aircraft_Type", "Destination", "STD", "BlockOff_Time", "TakeOff_Time", "Stand", "Data_Date"]].head())

# ============================================================
# 12. RINGKASAN AKHIR
# ============================================================

print("\n" + "="*70)
print(" RINGKASAN KARAKTERISTIK DATA MENTAH")
print("="*70)

print(f"""

KARAKTERISTIK DATA MENTAH               

│   RAW_ARRIVAL
│     Total baris          : {len(df_arr)}
│     Total kolom          : {len(df_arr.columns)}
│     Total missing        : {df_arr.isnull().sum().sum()}
│     Rentang tanggal      : {df_arr['Data_Date'].min().date()} - {df_arr['Data_Date'].max().date()}
│     Maskapai terbanyak   : {airline_arr.index[0]} ({airline_arr.iloc[0]} baris)
│     Hari terbanyak       : {day_arr.idxmax()} ({day_arr.max()} baris)
│     Bulan terbanyak      : {month_arr.idxmax()} ({month_arr.max()} baris) 
│                                                                 
│   RAW_DEPARTURE 
│     Total baris          : {len(df_dep)}
│     Total kolom          : {len(df_dep.columns)}
│     Total missing        : {df_dep.isnull().sum().sum()}
│     Rentang tanggal      : {df_dep['Data_Date'].min().date()} - {df_dep['Data_Date'].max().date()}
│     Maskapai terbanyak   : {airline_dep.index[0]} ({airline_dep.iloc[0]} baris)
│     Hari terbanyak       : {day_dep.idxmax()} ({day_dep.max()} baris)
│     Bulan terbanyak      : {month_dep.idxmax()} ({month_dep.max()} baris)

""")

print("="*70)
print(" ANALISIS KARAKTERISTIK DATA MENTAH SELESAI")

 ANALISIS KARAKTERISTIK DATA MENTAH

1. STRUKTUR DATA

 RAW_ARRIVAL:
   Jumlah baris : 3498
   Jumlah kolom : 20
   Nama kolom   : ['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Aircraft_Type', 'Origin', 'STA', 'Landing_Time', 'Onblock_Time', 'Stand', 'AVB', 'Adult_Pax', 'Child_Pax', 'Infant_Pax', 'Transit', 'Total_Pax', 'Seat', 'Cargo_Kg', 'Baggage_Kg', 'Data_Date', 'Table_Type']

 RAW_DEPARTURE:
   Jumlah baris : 3503
   Jumlah kolom : 20
   Nama kolom   : ['Flight_Number', 'Paired_Flight', 'Aircraft_Reg', 'Aircraft_Type', 'Destination', 'STD', 'BlockOff_Time', 'TakeOff_Time', 'Stand', 'AVB', 'Adult_Pax', 'Child_Pax', 'Infant_Pax', 'Transit', 'Total_Pax', 'Seat', 'Cargo_Kg', 'Baggage_Kg', 'Data_Date', 'Table_Type']

2. TIPE DATA

 RAW_ARRIVAL:
Flight_Number     object
Paired_Flight     object
Aircraft_Reg      object
Aircraft_Type     object
Origin            object
STA               object
Landing_Time      object
Onblock_Time      object
Stand             object
AVB           